# CDLE: Benchmark de Bibliotecas de Big Data em Python

Este notebook apresenta um estudo comparativo aprofundado entre três dos principais ecossistemas de manipulação de dados em Python: **Pandas**, **Dask** e **PySpark (Pandas-on-Spark/Koalas)**. O objetivo é analisar de forma rigorosa as diferenças de desempenho, comportamento de memória, escalabilidade e complexidade sintática ao realizar operações fundamentais em bases de dados de grande escala.

### Objetivos Académicos e Práticos:
1. **Identificar Gargalos de Desempenho (Bottlenecks):** Avaliar como cada biblioteca lida com operações intensivas de I/O, agregações e filtragens.
2. **Análise Sintática e de Paradigmas:** Confrontar a execução ansiosa (*eager*) com a execução preguiçosa (*lazy*).
3. **Casos de Uso Ideais:** Compreender em que cenários práticos cada ferramenta se destaca.
4. **Benchmarking em Ambiente GCP:** Executar as operações diretamente sobre o **Google Cloud Storage (GCS)** num cluster distribuído do **GCP Dataproc**, avaliando o comportamento sob restrições físicas de hardware.

### O Dataset:
Utilizaremos os dados públicos de viagens de táxi de Nova Iorque (**NYC Yellow Taxi Tripdata**), especificamente o mês de **Janeiro de 2009** no formato otimizado **Parquet**. Trata-se de um ficheiro de dimensão considerável (aproximadamente 14 milhões de registos), ideal para testar os limites de memória e processamento das ferramentas.

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# O caminho do ficheiro aponta diretamente para o Google Cloud Storage
file_path = 'gs://dataproc-staging-europe-southwest1-348488791616-f80l4tzf/notebooks/jupyter/yellow_tripdata_2009-01.parquet'

# Dicionários para guardar os tempos de execução
pandas_results = {}
dask_results = {}
pyspark_results = {}

---
## Parte 1: Pandas (O Padrão Eager e Single-Node)

O **Pandas** é a biblioteca de referência absoluta para ciência de dados em Python. No entanto, foi desenhado sob premissas específicas que determinam a sua viabilidade em Big Data:

### Características Principais:
* **Execução Eager (Ansiosa):** Cada linha de código é compilada e executada de imediato. Qualquer operação (leitura, filtro, agregação) gera imediatamente um novo objeto concreto na memória RAM.
* **Arquitetura Single-Node e Single-Threaded:** O Pandas opera estritamente num único computador e, devido ao *Global Interpreter Lock (GIL)* do Python, a maior parte das suas operações corre numa única thread (single-core).
* **Pegada de Memória (Memory Footprint):** Como regra geral, o Pandas requer entre **2 a 3 vezes mais memória RAM** do que o tamanho do dataset em disco para realizar operações de cópia, ordenação e alinhamento de índices. Isto torna-o altamente suscetível a falhas de *Out of Memory (OOM)* com datasets médios.

Nesta secção, medimos o desempenho básico do Pandas como a nossa linha de base (*baseline*) de comparação.

### 1.1 Read Data (Leitura do Parquet)

In [ ]:
start = time.time()
df_pd = pd.read_parquet(file_path)
pandas_results['1. Read'] = time.time() - start

print(f"Tempo de Leitura Pandas: {pandas_results['1. Read']:.3f} s")
display(df_pd.head(5))

### 1.2 Count (Contagem de Linhas)

In [ ]:
start = time.time()
total_rows_pd = len(df_pd)
pandas_results['2. Count'] = time.time() - start

print(f"Tempo de Contagem Pandas: {pandas_results['2. Count']:.3f} s (Total de viagens: {total_rows_pd})")

### 1.3 Value Counts (Frequências)
Contar quantas viagens foram processadas por cada fornecedor (`vendor_name`).

In [ ]:
start = time.time()
vc_pd = df_pd['vendor_name'].value_counts()
pandas_results['3. Value Counts'] = time.time() - start

print(f"Tempo de Value Counts Pandas: {pandas_results['3. Value Counts']:.3f} s")
display(vc_pd.to_frame())

### 1.4 GroupBy (Agrupamento e Agregação)
Agrupar pelo tipo de pagamento (`Payment_Type`) e calcular a tarifa média.

In [ ]:
start = time.time()
gb_pd = df_pd.groupby('Payment_Type')['Fare_Amt'].mean()
pandas_results['4. GroupBy'] = time.time() - start

print(f"Tempo de GroupBy Pandas: {pandas_results['4. GroupBy']:.3f} s")
display(gb_pd.to_frame())

### 1.5 Add Column (Mutação de Dados)
Criar uma nova coluna `Total_Calculated` adicionando a tarifa, gorjeta e portagens.

In [ ]:
start = time.time()
df_pd['Total_Calculated'] = df_pd['Fare_Amt'] + df_pd['Tip_Amt'] + df_pd['Tolls_Amt']
pandas_results['5. Add Column'] = time.time() - start

print(f"Tempo de Adição de Coluna Pandas: {pandas_results['5. Add Column']:.3f} s")
display(df_pd[['Fare_Amt', 'Tip_Amt', 'Tolls_Amt', 'Total_Calculated']].head(2))

### 1.6 Filter (Filtragem)
Encontrar todas as viagens que custaram estritamente mais de 10 dólares.

In [ ]:
start = time.time()
filtered_pd = df_pd[df_pd['Fare_Amt'] > 10]
pandas_results['6. Filter'] = time.time() - start

print(f"Tempo de Filtragem Pandas: {pandas_results['6. Filter']:.3f} s (Total filtrado: {len(filtered_pd)})")

Libertar memória do Pandas antes de iniciar o Dask

In [ ]:
import gc
del df_pd, filtered_pd
gc.collect()

---
## Parte 2: Dask (Computação Paralela e Out-of-Core baseada em Grafos)

O **Dask** é uma biblioteca de computação flexível e paralela que expande o ecossistema científico do Python (Pandas, NumPy, Scikit-Learn) para sistemas distribuídos e processamento que excede a memória RAM disponível (*Out-of-Core*).

### Como Funciona o Dask:
* **Arquitetura Particionada:** Um `dask.dataframe` é composto por múltiplos pequenos DataFrames do Pandas, divididos por partições de linhas. O Dask gere a execução de operações sobre cada partição de forma coordenada.
* **Execução Lazy (Preguiçosa):** Ao contrário do Pandas, o Dask não executa as operações imediatamente. Em vez disso, ele constrói um **Grafo Acíclico Dirigido (DAG)** que mapeia todas as tarefas necessárias. A computação real só é disparada quando invocamos uma ação explícita, como `.compute()`, `.head()` ou `len()`.
* **Otimização de Grafo:** O motor do Dask analisa o DAG para fundir operações (ex: ler o ficheiro e aplicar um filtro na mesma passagem), evitando leituras desnecessárias de colunas e poupando memória.
* **Gestão de Memória Estável:** Permite processar datasets muito maiores do que a memória RAM física através do processamento sequencial de partições e descarte automático de dados intermédios (*garbage collection*).

*(Nota: Para garantir estabilidade no nó Master do Dataproc de 16GB, configurámos o agendador do Dask para modo síncrono/sequencial, garantindo que o processamento de partições não gere picos acumulados de RAM)*

### 2.1 Read Data (Leitura do Parquet)

In [ ]:
# Monkey-patch para resolver incompatibilidade entre Dask antigo e Python 3.11+
try:
    import dask.utils
    original_derived_from = dask.utils.derived_from
    def safe_derived_from(*args, **kwargs):
        decorator = original_derived_from(*args, **kwargs)
        def safe_decorator(func):
            try:
                return decorator(func)
            except Exception:
                return func
        return safe_decorator
    dask.utils.derived_from = safe_derived_from
    import dask
    dask.config.set(scheduler='synchronous') # Execução sequencial para evitar OOM no Master node
except Exception as e:
    print(f"Aviso ao aplicar patch do Dask: {e}")

import dask.dataframe as dd

start = time.time()
df_dd = dd.read_parquet(file_path)
dask_results['1. Read'] = time.time() - start

print(f"Tempo de Leitura Dask: {dask_results['1. Read']:.3f} s (Note-se que a avaliação é Lazy)")

### 2.2 Count (Conta as linhas)

In [ ]:
start = time.time()
total_rows_dd = len(df_dd)
dask_results['2. Count'] = time.time() - start

print(f"Tempo de Contagem Dask: {dask_results['2. Count']:.3f} s")

### 2.3 Value Counts (Frequências)
Contar quantas viagens foram processadas por cada fornecedor (`vendor_name`).

In [ ]:
start = time.time()
# Temos de forçar o cálculo da distribuição com o .compute()
vc_dd = df_dd['vendor_name'].value_counts().compute()
dask_results['3. Value Counts'] = time.time() - start

print(f"Tempo de Value Counts Dask: {dask_results['3. Value Counts']:.3f} s")
display(vc_dd.to_frame())

### 2.4 GroupBy (Agrupamento e Agregação)
Agrupar pelo tipo de pagamento (`Payment_Type`) e calcular a tarifa média.

In [ ]:
start = time.time()
gb_dd = df_dd.groupby('Payment_Type')['Fare_Amt'].mean().compute()
dask_results['4. GroupBy'] = time.time() - start

print(f"Tempo de GroupBy Dask: {dask_results['4. GroupBy']:.3f} s")
display(gb_dd.to_frame())

### 2.5 Add Column (Mutação de Dados)
Criar uma nova coluna `Total_Calculated` adicionando a tarifa, gorjeta e portagens.

In [ ]:
start = time.time()
df_dd['Total_Calculated'] = df_dd['Fare_Amt'] + df_dd['Tip_Amt'] + df_dd['Tolls_Amt']
preview_dd = df_dd['Total_Calculated'].head() # Usamos o head() para simular o trigger da ação
dask_results['5. Add Column'] = time.time() - start

print(f"Tempo de Adição de Coluna Dask: {dask_results['5. Add Column']:.3f} s")

### 2.6 Filter (Filtragem)
Encontrar todas as viagens que custaram estritamente mais de 10 dólares.

In [ ]:
start = time.time()
filtered_dd = df_dd[df_dd['Fare_Amt'] > 10].head()
dask_results['6. Filter'] = time.time() - start

print(f"Tempo de Filtragem Dask: {dask_results['6. Filter']:.3f} s")

Libertar memória do Dask antes de iniciar o PySpark, para não haver crash do clutcher.

In [ ]:
import gc
del df_dd, filtered_dd
gc.collect()

---
## Parte 3: PySpark (Processamento Massivo Distribuído - Spark Engine)

O **Apache Spark** é a plataforma de computação distribuída padrão da indústria para processamento de Big Data. O **PySpark** fornece a interface Python para o motor do Spark, e através da API **Pandas-on-Spark** (antigo Koalas), permite executar código com sintaxe Pandas diretamente sobre a arquitetura do Spark.

### O Motor do PySpark:
* **Execução Distribuída Massiva (Master-Worker):** O Spark foi desenhado desde a raiz para funcionar em clusters (como o GCP Dataproc). O nó Master planeia a execução e distribui as partições de dados para múltiplos nós Workers, que processam a informação em paralelo na JVM (Java Virtual Machine).
* **Lazy Evaluation Avançada (Catalyst Optimizer):** Tal como o Dask, o Spark é lazy. No entanto, ele possui o otimizador *Catalyst*, que traduz o código em planos de execução lógicos e físicos altamente otimizados antes de iniciar qualquer computação.
* **Resiliência e Tolerância a Falhas:** Os dados são mantidos em estruturas resilientes. Se um nó worker falhar a meio de uma computação, o Spark reconstrói automaticamente a partição perdida a partir do grafo de linhagem (*lineage graph*).
* **API Pandas-on-Spark (`pyspark.pandas`):** Permite que cientistas de dados familiarizados com o Pandas escalem os seus códigos para petabytes de dados sem reescrever a lógica de manipulação.

In [ ]:
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'
from pyspark.sql import SparkSession
import pyspark.pandas as ps

# Inicializar o Spark (no Dataproc, ele utiliza o YARN por defeito)
spark = SparkSession.builder \
    .appName('CDLE-Benchmark') \
    .config('spark.sql.ansi.enabled', 'false') \
    .getOrCreate()

start = time.time()
df_ps = ps.read_parquet(file_path)
pyspark_results['1. Read'] = time.time() - start

print(f"Tempo de Leitura PySpark: {pyspark_results['1. Read']:.3f} s")
display(df_ps.head(2))

In [ ]:
start = time.time()
total_rows_ps = len(df_ps)
pyspark_results['2. Count'] = time.time() - start

print(f"Tempo de Contagem PySpark: {pyspark_results['2. Count']:.3f} s")

In [ ]:
start = time.time()
vc_ps = df_ps['vendor_name'].value_counts()
pyspark_results['3. Value Counts'] = time.time() - start

print(f"Tempo de Value Counts PySpark: {pyspark_results['3. Value Counts']:.3f} s")
display(vc_ps.to_frame())

In [ ]:
start = time.time()
gb_ps = df_ps.groupby('Payment_Type')['Fare_Amt'].mean()
pyspark_results['4. GroupBy'] = time.time() - start

print(f"Tempo de GroupBy PySpark: {pyspark_results['4. GroupBy']:.3f} s")
display(gb_ps.to_frame())

In [ ]:
start = time.time()
df_ps['Total_Calculated'] = df_ps['Fare_Amt'] + df_ps['Tip_Amt'] + df_ps['Tolls_Amt']
pyspark_results['5. Add Column'] = time.time() - start

print(f"Tempo de Adição de Coluna PySpark: {pyspark_results['5. Add Column']:.3f} s")

In [ ]:
start = time.time()
filtered_ps = df_ps[df_ps['Fare_Amt'] > 10].head()
pyspark_results['6. Filter'] = time.time() - start

print(f"Tempo de Filtragem PySpark: {pyspark_results['6. Filter']:.3f} s")

# Parar SparkSession e libertar memória para o gráfico final
import gc
spark.stop()
del df_ps, filtered_ps
gc.collect()


---
## Parte 4: Análise e Comparação de Desempenho
Por fim, reunimos os tempos de execução guardados em todas as secções numa única tabela Pandas para extrair conclusões.

In [ ]:
results_df = pd.DataFrame({
    'Pandas (s)': pandas_results,
    'Dask (s)': dask_results,
    'PySpark (s)': pyspark_results
})

print('=== Resumo Comparativo dos Tempos de Execução (Segundos) ===')
display(results_df.round(3))

ax = results_df.plot(kind='bar', figsize=(14, 7), rot=45, colormap='viridis')
plt.title('Comparação de Performance (Menos Segundos = Mais Rápido)', fontsize=16)
plt.ylabel('Tempo de Execução (segundos)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Biblioteca')
plt.tight_layout()
plt.show()